# 02 — Exploratory Statistics
**Group P33** — Mika Ohayon, Akshaya Somasundaram, Harini Champooranan

| Notebook | Contents |
|---|---|
| `01_Data_Prep_Cleaning_Merge.ipynb` | Clean the raw BLS 2025, BLS 2022, and AIOE files, then merge them into `finalized_data.csv` |
| `02_Exploratory_Statistics.ipynb` | Descriptive statistics, correlations, model-form checks, and first look at 2022 → 2025 change |
| `03_Findings_Visuals_Models.ipynb` | Interactive visuals 1–10, machine learning models, results, and conclusions |

**Run them in order (01 → 02 → 03).** Notebooks 02 and 03 load `finalized_data.csv`, which notebook 01 creates.

### This notebook
Before building charts and models, we check what the numbers look like:
1. Descriptive statistics for the 670 occupations
2. How the two skill tiers differ
3. Correlations and whether a straight line is the right model form
4. A first look at change from 2022 to 2025

Requires `finalized_data.csv` from notebook 01.


In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 140)

In [2]:
df = pd.read_csv('finalized_data.csv', dtype={'soc_major': str, 'OCC_CODE': str})

HI, LO = 'Higher-skill (SOC 11–29)', 'Lower-skill (SOC 31–53)'
BAND_LABELS = ['Bottom 20%', '20–40%', '40–60%', '60–80%', 'Top 20%']
BAND_ORDER = BAND_LABELS[::-1]
BIN_ORDER = [f'{s:+.1f} to {s + 0.5:+.1f}' for s in sorted(df['aioe_bin_start'].unique())]

print('Loaded finalized_data.csv:', df.shape)
print(df['skill_tier'].value_counts().to_string())

Loaded finalized_data.csv: (670, 21)
skill_tier
Lower-skill (SOC 31–53)     393
Higher-skill (SOC 11–29)    277


---
## 1. Descriptive statistics

In [3]:
desc = df[['AIOE', 'A_MEDIAN', 'TOT_EMP']].describe().T
desc['skew'] = df[['AIOE', 'A_MEDIAN', 'TOT_EMP']].skew()
desc.round(2)

,count,mean,std,min,25%,50%,75%,max,skew
AIOE,670.0,-0.02,1.01,-2.11,-0.89,-0.12,1.02,1.53,-0.01
A_MEDIAN,670.0,69764.34,35040.87,30890.00,47052.50,60250.00,80645.00,352220.00,2.71
TOT_EMP,670.0,183465.42,418070.74,220.00,14102.50,44705.00,150180.00,3897860.00,5.00


> **Key takeaways**
>
> - **AIOE** is standardized: mean ≈ 0, SD ≈ 1, range −2.1 to +1.5. It is symmetric overall (skew ≈ 0) but bunched near the top, with a distinct cluster of highly exposed jobs.
> - **Median wage** is right-skewed (skew ≈ 2.7): typical job $60k, mean $70k, max $352k. That's why we use **log wage** in the models.
> - **Employment** is very right-skewed (skew ≈ 5): typical job about 45k workers, while the largest (retail salespersons) has 3.9M. That's why we use **log employment**.

---
## 2. Higher-skill vs lower-skill occupations

In [4]:
tier = (df.groupby('skill_tier')
          .agg(occupations=('AIOE', 'size'),
               mean_AIOE=('AIOE', 'mean'),
               median_AIOE=('AIOE', 'median'),
               share_above_avg_exposure=('AIOE', lambda s: (s > 0).mean()),
               median_wage=('A_MEDIAN', 'median'),
               total_workers=('TOT_EMP', 'sum')))
tier.round(2)

,occupations,mean_AIOE,median_AIOE,share_above_avg_exposure,median_wage,total_workers
skill_tier,,,,,,
Higher-skill (SOC 11–29),277,0.78,0.99,0.86,81100.0,42216180.0
Lower-skill (SOC 31–53),393,-0.58,-0.76,0.20,49230.0,80705650.0


> **Key takeaways**
>
> - **277 higher-skill** occupations (SOC 11–29: managers, professionals, tech, healthcare) and **393 lower-skill** (SOC 31–53: service, sales, office, trades, production, transport).
> - Higher-skill jobs are much **more exposed** (mean AIOE **+0.78** vs **−0.58**). 86% are above average, compared with 20% of lower-skill jobs.
> - Higher-skill jobs **pay more** (median **$81k vs $49k**) but employ **fewer people** (42M vs 81M workers).
> - Skill and exposure overlap heavily, so any model of exposure must account for skill. That's why we use an **interaction term**.

---
## 3. Correlations and model form (2025)

In [5]:
corr_cols = ['AIOE', 'TOT_EMP', 'A_MEDIAN', 'A_PCT10', 'A_PCT90']
print('Pearson correlations:')
print(df[corr_cols].corr().round(2).to_string())


def adj_r2(x, y, deg):
    coefs = np.polyfit(x, y, deg)
    pred = np.polyval(coefs, x)
    r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    n = len(y)
    return 1 - (1 - r2) * (n - 1) / (n - deg - 1)


print('\nAdjusted R² by polynomial degree:')
for target in ['A_MEDIAN', 'TOT_EMP']:
    vals = {deg: round(adj_r2(df['AIOE'], df[target], deg), 3) for deg in (1, 2, 3)}
    print(f'  AIOE → {target:8s}  linear {vals[1]}  quadratic {vals[2]}  cubic {vals[3]}')

Pearson correlations:
          AIOE  TOT_EMP  A_MEDIAN  A_PCT10  A_PCT90
AIOE      1.00     0.02      0.45     0.45     0.44
TOT_EMP   0.02     1.00     -0.07    -0.07    -0.05
A_MEDIAN  0.45    -0.07      1.00     0.90     0.87
A_PCT10   0.45    -0.07      0.90     1.00     0.73
A_PCT90   0.44    -0.05      0.87     0.73     1.00

Adjusted R² by polynomial degree:
  AIOE → A_MEDIAN  linear 0.2  quadratic 0.2  cubic 0.199
  AIOE → TOT_EMP   linear -0.001  quadratic -0.002  cubic 0.001


> **Key takeaways**
>
> - AI exposure is **moderately positively** correlated with pay (**r ≈ 0.45**): more exposed jobs tend to pay more.
> - Exposure is **not** related to how many people do a job (**r ≈ 0.02**).
> - The wage measures are strongly correlated with each other (0.73–0.90), as expected, since they describe the same pay distribution. We use the median.
> - Quadratic and cubic fits are no better than a straight line (adjusted R² 0.200 in all three), so **linear models are enough**.

In [6]:
# Does the exposure–wage relationship differ by tier?
for t, g in df.groupby('skill_tier'):
    print(f"{t}:  r(AIOE, log wage) = {g['AIOE'].corr(g['log_wage']):.2f}   "
          f"r(AIOE, log employment) = {g['AIOE'].corr(g['log_emp']):.2f}")

Higher-skill (SOC 11–29):  r(AIOE, log wage) = 0.32   r(AIOE, log employment) = 0.05
Lower-skill (SOC 31–53):  r(AIOE, log wage) = 0.13   r(AIOE, log employment) = 0.15


> **Key takeaways**
>
> - Inside the **higher-skill** tier the exposure–pay link is clearly stronger (**r = 0.32**) than in the **lower-skill** tier (**r = 0.13**).
> - That difference is the **interaction** the models in notebook 03 test formally.

---
## 4. First look at change: 2022 → 2025
Everything above describes **levels** in one year: who is exposed. To test displacement we need **change**.
- **Wage growth** = log(median wage 2025 / median wage 2022)
- **Employment growth** = log(workers 2025 / workers 2022)

We drop occupations with under 1,000 workers in either year, because their estimates swing a lot from sampling noise alone.

In [7]:
MIN_EMP = 1000
chg = df.dropna(subset=['A_MEDIAN_22', 'TOT_EMP_22']).copy()
chg = chg[(chg['TOT_EMP_22'] >= MIN_EMP) & (chg['TOT_EMP'] >= MIN_EMP)].copy()

chg['wage_growth'] = np.log(chg['A_MEDIAN'] / chg['A_MEDIAN_22'])
chg['emp_growth'] = np.log(chg['TOT_EMP'] / chg['TOT_EMP_22'])
chg['wage_growth_pct'] = 100 * (np.exp(chg['wage_growth']) - 1)
chg['emp_growth_pct'] = 100 * (np.exp(chg['emp_growth']) - 1)
chg['rel_wage_growth'] = 100 * (chg['wage_growth'] - chg['wage_growth'].median())
chg['rel_emp_growth'] = 100 * (chg['emp_growth'] - chg['emp_growth'].median())
chg['log_wage_22'] = np.log(chg['A_MEDIAN_22'])
chg['log_emp_22'] = np.log10(chg['TOT_EMP_22'])
chg['aioe_quintile'] = pd.qcut(chg['AIOE'], 5, labels=['Q1 (least)', 'Q2', 'Q3', 'Q4', 'Q5 (most)']).astype(str)

print(f'Change-analysis sample: {len(chg)} occupations')
print(f"Typical (median) occupation: wages {chg['wage_growth_pct'].median():+.1f}%, "
      f"employment {chg['emp_growth_pct'].median():+.1f}% from 2022 to 2025")
print('\nCorrelation of AIOE with wage growth:       r =', round(chg['AIOE'].corr(chg['wage_growth']), 3))
print('Correlation of AIOE with employment growth: r =', round(chg['AIOE'].corr(chg['emp_growth']), 3))

Change-analysis sample: 661 occupations
Typical (median) occupation: wages +13.1%, employment +2.9% from 2022 to 2025

Correlation of AIOE with wage growth:       r = -0.106
Correlation of AIOE with employment growth: r = 0.119


> **Key takeaways**
>
> - **661 occupations** are in both years with enough workers to compare.
> - From 2022 to 2025 the typical job's pay rose **13.1%** (mostly inflation), and employment rose **2.9%**.
> - Exposure barely relates to growth overall (r ≈ −0.11 for wages, +0.12 for employment), so we need to look **by skill tier**.

In [8]:
# Summary table: median growth by skill tier × AIOE quintile
summary = (chg.groupby(['skill_tier', 'aioe_quintile'])
              .agg(occupations=('AIOE', 'size'),
                   median_wage_growth_pct=('wage_growth_pct', 'median'),
                   median_emp_growth_pct=('emp_growth_pct', 'median'),
                   workers_2025=('TOT_EMP', 'sum'))
              .round(1))
summary

occupations  median_wage_growth_pct  median_emp_growth_pct  workers_2025
skill_tier               aioe_quintile                                                                          
Higher-skill (SOC 11–29) Q1 (least)               3                     8.5                   -4.6      183150.0
                         Q2                       6                    18.8                   -0.6     1046090.0
                         Q3                      60                    15.8                    6.6     8354040.0
                         Q4                      95                    11.4                    6.5    15918530.0
                         Q5 (most)              109                    11.0                    6.3    16707490.0
Lower-skill (SOC 31–53)  Q1 (least)             130                    13.9                   -0.8    25167380.0
                         Q2                     126                    14.4                    1.7    16978250.0
                         Q3                      72                    13.9                    4.8    19603980.0
                         Q4                      37                    14.6                   -5.5    14606770.0
                         Q5 (most)               23                    11.8                   -9.7     4345550.0

> **Key takeaways**
>
> - **Pay growth shows no clear pattern by exposure level**: the most exposed groups (Q5) got about the same raise (11–12%) as the typical job (13%).
> - **Employment is where the tiers split:** the most exposed **lower-skill** jobs shrank (**−5.5% in Q4, −9.7% in Q5**), while the most exposed **higher-skill** jobs grew (**+6.3% to +6.6%**).
> - Note: the two least-exposed higher-skill groups have only 3 and 6 occupations each, so ignore those rows.

---
### Summary for the next notebook
- Exposure is concentrated in **higher-paid, higher-skill** jobs, so models need a **skill × exposure interaction**.
- **Linear** models are an appropriate starting point. Wages and employment should be **logged**.
- Early evidence: **pay** growth doesn't depend on exposure, but **employment** in lower-skill exposed jobs may be shrinking.

**Next:** open `03_Findings_Visuals_Models.ipynb` for the interactive visuals and machine learning models that test these patterns.